# Questão 14


### Recursivo com Memoização

In [1]:
def seam_carving_recursive(E):
    n, m = len(E), len(E[0])

    def seam_min_cost(i, j):
        if j < 0 or j >= m:
            return float('inf')
        if i == n - 1:
            return E[i][j]
        return E[i][j] + min(
            seam_min_cost(i + 1, j - 1),
            seam_min_cost(i + 1, j),
            seam_min_cost(i + 1, j + 1)
        )

    def reconstruct_path(i, j):
        path = [(i, j)]
        while i < n - 1:
            options = []
            for dj in [-1, 0, 1]:
                nj = j + dj
                if 0 <= nj < m:
                    cost = seam_min_cost(i + 1, nj)
                    options.append((cost, nj))
            _, j = min(options)
            i += 1
            path.append((i, j))
        return path

    best_cost = float('inf')
    best_j = -1
    for j in range(m):
        cost = seam_min_cost(0, j)
        if cost < best_cost:
            best_cost = cost
            best_j = j

    best_path = reconstruct_path(0, best_j)
    return best_cost, best_path


### Iterativo

In [2]:
def seam_carving_iterative(E):
    n, m = len(E), len(E[0])
    dp = [[0] * m for _ in range(n)]
    path = [[-1] * m for _ in range(n)]

    for j in range(m):
        dp[0][j] = E[0][j]

    for i in range(1, n):
        for j in range(m):
            min_val = float('inf')
            prev_j = -1
            for dj in [-1, 0, 1]:
                pj = j + dj
                if 0 <= pj < m:
                    if dp[i - 1][pj] < min_val:
                        min_val = dp[i - 1][pj]
                        prev_j = pj
            dp[i][j] = E[i][j] + min_val
            path[i][j] = prev_j

    min_cost = min(dp[-1])
    end_j = dp[-1].index(min_cost)

    seam = []
    j = end_j
    for i in range(n - 1, -1, -1):
        seam.append((i, j))
        j = path[i][j]
    seam.reverse()

    return min_cost, seam


### Testes

In [3]:
import time
import matplotlib as plt
import numpy as np

In [4]:
def run_experiments(k=20, largura_m=30, m_trials=10):
    sizes = np.linspace(10, 200, k, dtype=int)
    tempos_rec, tempos_it = [], []

    for n in sizes:
        t_rec, t_it = [], []
        for _ in range(m_trials):
            E = np.random.randint(0, 256, size=(n, largura_m)).tolist()

            # Recursivo
            start = time.perf_counter()
            seam_carving_recursive(tuple(map(tuple, E)))
            t_rec.append(time.perf_counter() - start)

            # Iterativo
            start = time.perf_counter()
            seam_carving_iterative(E)
            t_it.append(time.perf_counter() - start)

        tempos_rec.append((np.mean(t_rec), np.std(t_rec)))
        tempos_it.append((np.mean(t_it), np.std(t_it)))

    return sizes, tempos_rec, tempos_it

def generate_grapic(sizes, tempos_rec, tempos_it, nome_arquivo="seam_carving_comparacao.csv"):
    mean_rec, std_rec = zip(*tempos_rec)
    mean_it, std_it = zip(*tempos_it)

    # Gráfico
    plt.figure(figsize=(12, 6))
    plt.errorbar(sizes, mean_rec, yerr=std_rec, label='Recursivo com Memoização', fmt='-o')
    plt.errorbar(sizes, mean_it, yerr=std_it, label='Iterativo', fmt='-s')
    plt.xlabel('Altura da matriz (n)')
    plt.ylabel('Tempo médio de execução (s)')
    plt.title('Comparação de desempenho: Seam Carving Recursivo vs Iterativo')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


sizes, tempos_rec, tempos_it = run_experiments(k=20, largura_m=30, m_trials=10)
generate_grapic(sizes, tempos_rec, tempos_it)


KeyboardInterrupt: 